In [ ]:
import math
import collections
import re
import numpy as np

def calculate_metrics(text_list):
    """计算一组文本的平均 TTR 和信息熵"""
    all_ttrs = []
    all_entropies = []
    
    for text in text_list:
        words = re.findall(r'\w+', text.lower())
        if len(words) < 5: continue  # 过滤过短的碎片
        
        # TTR: 词汇丰富度
        ttr = len(set(words)) / len(words)
        all_ttrs.append(ttr)
        
        # Shannon Entropy: 信息密度
        counts = collections.Counter(words)
        total = len(words)
        entropy = -sum((c/total) * math.log2(c/total) for c in counts.values())
        all_entropies.append(entropy)
        
    return np.mean(all_ttrs), np.mean(all_entropies)

In [ ]:
def strip_enron_global_clean(raw_text):
    """
    全局扫描并剔除所有符合邮件头、转发块、以及散落在正文中的元数据短句
    """
    # 1. 定义需要严格匹配开头的关键字（通常是元数据）
    strict_header_keys = {
        'Date:', 'From:', 'To:', 'Subject:', 'Mime-Version:', 
        'Content-Type:', 'Content-Transfer-Encoding:', 'X-From:', 
        'X-To:', 'X-cc:', 'X-bcc:', 'X-Folder:', 'X-Origin:', 'X-FileName:',
        'cc:', 'bcc:', 'Re:', 'FW:', 'Fwd:', 'Sent:'
    }

    # 2. 预编译更复杂的模式（处理时间戳、转发分割线等短句）
    # 匹配类似 "05/22/99 11:12 AM" 或 "----- Forwarded by" 或 "----------------------"
    patterns_to_remove = [
        re.compile(r'^\d{2}/\d{2}/\d{2,4}\s+\d{2}:\d{2}\s+[AP]M'), # 时间戳行
        re.compile(r'^-+.*Forwarded by.*-+', re.IGNORECASE),       # 转发分割线
        re.compile(r'^Sent by:.*', re.IGNORECASE),                 # 发送人标记
        re.compile(r'^-{3,}.*'),                                   # 长横线
    ]

    lines = raw_text.split('\n')
    clean_lines = []

    for line in lines:
        stripped_line = line.strip()
        
        # 逻辑 A: 跳过空行
        if not stripped_line:
            continue
            
        # 逻辑 B: 检查是否以任意元数据关键字开头
        if any(stripped_line.startswith(key) for key in strict_header_keys):
            # print(f"[Removed Key]: {stripped_line}")
            continue
            
        # 逻辑 C: 检查是否匹配复杂的短句模式
        if any(pattern.match(stripped_line) for pattern in patterns_to_remove):
            # print(f"[Removed Pattern]: {stripped_line}")
            continue
            
        # 逻辑 D: 额外的启发式过滤——如果一行太短且包含冒号（例如 "Mark Taylor:"）
        # 这种行往往是残留的元数据。
        if len(stripped_line) < 40 and ":" in stripped_line:
            # 排除掉正常的句子（通常以标点结束）
            if not stripped_line.endswith(('.', '?', '!', '"')):
                # print(f"[Removed Short Metadata]: {stripped_line}")
                continue

        # 剩下的才是真正的正文
        clean_lines.append(line)

    # 重新组合
    return "\n".join(clean_lines).strip()

In [ ]:
from langchain_community.vectorstores import Chroma

ModuleNotFoundError: No module named 'langchain_core'

In [ ]:
db_fiqa = Chroma(persist_directory="./retrieval_stores/fiqa/bge-large-en-v1.5/chroma")
# 获取 FiQA 样本进行量化
fiqa_docs = db_fiqa.get(limit=1000)['documents']
fiqa_ttr, fiqa_ent = calculate_metrics(fiqa_docs)

print(f"FiQA -> TTR: {fiqa_ttr:.4f}, Entropy: {fiqa_ent:.4f}")

FiQA -> TTR: 0.7011, Entropy: 5.8880


In [ ]:
db_enronmail = Chroma(persist_directory="./retrieval_stores/enronmail/bge-large-en-v1.5/chroma")
# 获取 EnronMail 样本进行量化
enronmail_docs = db_enronmail.get(limit=1000)['documents']
enronmail_ttr, enronmail_ent = calculate_metrics(enronmail_docs)

print(f"EnronMail -> TTR: {enronmail_ttr:.4f}, Entropy: {enronmail_ent:.4f}")

EnronMail -> TTR: 0.5445, Entropy: 6.9466


In [ ]:
db_enronmail = Chroma(persist_directory="./retrieval_stores/enronmail/bge-large-en-v1.5/chroma")
# 获取 EnronMail 样本进行量化
enronmail_docs = db_enronmail.get(limit=1000)['documents']
for i in range(len(enronmail_docs)):
    enronmail_docs[i] = strip_enron_global_clean(enronmail_docs[i])
enronmail_ttr, enronmail_ent = calculate_metrics(enronmail_docs)

print(f"EnronMail -> TTR: {enronmail_ttr:.4f}, Entropy: {enronmail_ent:.4f}")

EnronMail -> TTR: 0.6073, Entropy: 6.3232


In [ ]:
db_scifact = Chroma(persist_directory="./retrieval_stores/scifact/bge-large-en-v1.5/chroma")
# 获取 FiQA 样本进行量化
scifact_docs = db_scifact.get(limit=1000)['documents']
scifact_ttr, scifact_ent = calculate_metrics(scifact_docs)

print(f"SciFact -> TTR: {scifact_ttr:.4f}, Entropy: {scifact_ent:.4f}")

SciFact -> TTR: 0.5718, Entropy: 6.4355


In [7]:
db_nfcorpus = Chroma(persist_directory="./retrieval_stores/nfcorpus/bge-large-en-v1.5/chroma")
# 获取 NFCorpus 样本进行量化
nfcorpus_docs = db_nfcorpus.get(limit=1000)['documents']
nfcorpus_ttr, nfcorpus_ent = calculate_metrics(nfcorpus_docs)

print(f"NFCorpus -> TTR: {nfcorpus_ttr:.4f}, Entropy: {nfcorpus_ent:.4f}")

NFCorpus -> TTR: 0.5516, Entropy: 6.5273
